# AutoLab — étape 6 : validation UMA sur Colab (GPU T4)

Ce notebook remplace temporairement Modal. Il fait, dans l'ordre :

| Phase | Ce qui est fait |
|---|---|
| A | clone le dépôt public et l'installe (`pip install -e .`, `fairchem-core==2.24.0`) |
| B | télécharge le checkpoint UMA `uma-s-1p2` (≈ 2,3 Go, dans le cloud Google, jamais sur le PC) |
| C | relaxe LiCl (test de vérité : a = 5,10–5,18 Å) — inclut la compilation du modèle |
| D | tests de vérité UMA : E_hull Li₂S, LiS rocksalt fictif, calibration Li₃PS₄ (≈ 50 relaxations Li-P-S) |
| D+ | calibration détaillée : E_hull UMA de toutes les références Li-P-S comparés à Materials Project (aucun nouveau calcul) |
| E | vitesse de la dynamique moléculaire (Li₃PS₄, 128 atomes, 800 K, précision normale) |
| Fin | rapport unique + fichier `autolab_step6_results.zip` à télécharger |

**Avant de lancer**
1. *Exécution → Modifier le type d'exécution →* **GPU T4**.
2. Panneau **Secrets** (icône clé, à gauche) : `HF_TOKEN` et `MP_API_KEY`, avec **Accès au notebook** activé. Les valeurs ne sont jamais affichées.
3. *Exécution → Tout exécuter.* Durée prévue : 25 à 40 minutes.

Chaque cellule capture ses erreurs : le notebook va jusqu'au bout même si une étape échoue. Copier le rapport final (dernière cellule) et télécharger le zip.

In [ ]:
# Helpers: every step records its status and duration; errors never stop the notebook.
import json, os, re, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/hkdebendo/autolab.git"
REPO = Path("/content/autolab")
WORK = Path("/content/autolab_run")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = REPO / ".cache"
T_START = time.time()
REPORT = {"steps": {}, "results": {}}
# Subprocesses get a clean interpreter (fresh imports after pip install) and the local UMA backend.
ENV = dict(os.environ, SIM_BACKEND="local", PYTHONUNBUFFERED="1", AUTOLAB_CACHE_DIR=str(CACHE))


def save_report():
    (WORK / "report.json").write_text(json.dumps(REPORT, indent=2, default=str), encoding="utf-8")


def record(step, status, seconds, **info):
    REPORT["steps"][step] = {"status": status, "seconds": round(seconds, 1), **info}
    save_report()
    print(f"\n[{step}] {status} en {seconds / 60:.1f} min")


def sh(step, cmd, cwd=REPO):
    """Run a command, stream its output live and into WORK/<step>.log; never raises."""
    t0 = time.time()
    try:
        with open(WORK / f"{step}.log", "w", encoding="utf-8") as log:
            proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=ENV, text=True,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            try:
                for line in proc.stdout:
                    print(line, end="")
                    log.write(line)
                code = proc.wait()
            except KeyboardInterrupt:
                proc.kill()
                record(step, "interrompu", time.time() - t0)
                return False
        record(step, "ok" if code == 0 else f"échec (code {code})", time.time() - t0)
        return code == 0
    except Exception as e:
        record(step, f"erreur : {type(e).__name__}: {e}", time.time() - t0)
        return False


def py(step, source):
    """Run a Python snippet in a fresh process from the repository folder."""
    script = WORK / f"{step}.py"
    script.write_text(source, encoding="utf-8")
    return sh(step, [sys.executable, script])


def load(name):
    p = WORK / f"{name}.json"
    return json.loads(p.read_text()) if p.exists() else None


def relax_cache():
    """All cached UMA relaxations (one JSON file each)."""
    folder = CACHE / "relax"
    out = []
    for p in sorted(folder.glob("*.json")) if folder.exists() else []:
        try:
            d = json.loads(p.read_text())
            d["mtime"] = p.stat().st_mtime
            out.append(d)
        except Exception:
            pass
    return out


print("Helpers prêts.")

In [ ]:
# GPU check
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "aucun GPU"
except Exception as e:
    gpu = f"aucun GPU ({type(e).__name__})"
REPORT["gpu"] = gpu
REPORT["python"] = sys.version.split()[0]
save_report()
print("GPU :", gpu, "| Python", REPORT["python"])
if "T4" not in gpu:
    print("⚠️ Pas de T4 : changer le type d'exécution (GPU T4), sinon les phases B à E échoueront ou ne seront pas comparables.")

In [ ]:
# Keys from Colab Secrets: only their presence is ever printed.
REPORT["secrets"] = {}
try:
    from google.colab import userdata
except Exception:
    userdata = None
for name in ("HF_TOKEN", "MP_API_KEY"):
    present, why = False, ""
    try:
        value = userdata.get(name) if userdata else os.environ.get(name, "")
        if value and value.strip():
            ENV[name] = value.strip()
            present = True
    except Exception as e:
        why = f" ({type(e).__name__} : secret absent ou accès au notebook non activé)"
    value = None
    REPORT["secrets"][name] = "présent" if present else "absent"
    print(f"{name} : {'présent' if present else 'ABSENT' + why}")
save_report()

In [ ]:
# Phase A: clone (or update) the public repository and install it.
if (REPO / ".git").exists():
    ok = sh("A_clone", ["git", "-C", REPO, "pull", "--ff-only"], cwd="/content")
else:
    ok = sh("A_clone", ["git", "clone", "--depth", "1", REPO_URL, REPO], cwd="/content")
if ok:
    ok = sh("A_install", [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev]", "fairchem-core==2.24.0"])
try:
    REPORT["commit"] = subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip()
except Exception as e:
    REPORT["commit"] = f"inconnu ({type(e).__name__})"
py("A_versions", r"""
import json, torch
from importlib.metadata import version
out = {p: version(p) for p in ("fairchem-core", "torch", "ase", "pymatgen", "numpy")}
out["cuda"] = torch.cuda.is_available()
out["gpu"] = torch.cuda.get_device_name(0) if out["cuda"] else None
open("/content/autolab_run/A_versions.json", "w").write(json.dumps(out))
print(out)
""")
REPORT["versions"] = load("A_versions")
save_report()
print("Commit testé :", REPORT["commit"])

In [ ]:
# Phase B: download the UMA checkpoint once (Hugging Face, gated repo facebook/UMA, needs HF_TOKEN).
py("B_download", r"""
import json, os, time
t0 = time.perf_counter()
from autolab.sim import uma
path = uma.download_checkpoint()
out = {"model": "uma-s-1p2", "size_gb": round(os.path.getsize(path) / 1e9, 2), "seconds": round(time.perf_counter() - t0, 1)}
open("/content/autolab_run/B_download.json", "w").write(json.dumps(out))
print(out)
""")
REPORT["results"]["download"] = load("B_download")
save_report()

In [ ]:
# Phase C: LiCl truth test, alone first (includes the one-time model compilation).
PYTEST = [sys.executable, "-m", "pytest", "tests/test_uma_gpu.py", "--gpu", "-s", "-rA", "-p", "no:cacheprovider"]
sh("C_licl", PYTEST + ["-k", "licl", f"--junitxml={WORK}/C_licl.xml"])

In [ ]:
# Phase D: the other UMA truth tests (Li-S and Li-P-S references relaxed by UMA, cached).
n_before = len(relax_cache())
sh("D_hull", PYTEST + ["-k", "not licl", f"--junitxml={WORK}/D_hull.xml"])
REPORT["steps"]["D_hull"]["new_relaxations"] = len(relax_cache()) - n_before
save_report()
print("Nouvelles relaxations :", REPORT["steps"]["D_hull"]["new_relaxations"])

In [ ]:
# Phase D+: detailed Li-P-S calibration, UMA E_hull vs Materials Project, from the cache only
# (max_new=0: if a reference is missing, it stops instead of computing).
py("D_calibration", r"""
import json
from statistics import mean
from autolab.analysis.stability import e_hull, make_entry, phase_diagram
from autolab.data.mp import get_references
from autolab.sim.backend import relax_structures

refs = get_references(["Li", "P", "S"])
res = relax_structures([r["structure"] for r in refs], max_new=0)
pairs = [(r, make_entry(r["structure"].composition, x["energy"], r["material_id"], "uma"))
         for r, x in zip(refs, res) if x["converged"]]
entries = [e for _, e in pairs]
pd = phase_diagram(entries)
rows = []
for r, e in pairs:
    out = e_hull(e, entries, pd)
    rows.append({"id": r["material_id"], "formula": e.composition.reduced_formula, "nsites": r["nsites"],
                 "uma": round(out["e_hull"], 3), "mp": round(r["mp_energy_above_hull"], 3)})
rows.sort(key=lambda d: (d["formula"], d["mp"]))
summary = {
    "n_refs": len(refs),
    "not_converged": [r["material_id"] for r, x in zip(refs, res) if not x["converged"]],
    "mae_vs_mp": round(mean(abs(d["uma"] - d["mp"]) for d in rows), 4),
    "rows": rows,
}
open("/content/autolab_run/D_calibration.json", "w").write(json.dumps(summary))
for d in rows:
    print(f"{d['formula']:<10} {d['id']:<12} {d['nsites']:>3} at.  UMA {d['uma']:.3f}  MP {d['mp']:.3f}")
print("Écart moyen |UMA - MP| :", summary["mae_vs_mp"], "eV/atome ; non convergées :", summary["not_converged"])
""")
REPORT["results"]["calibration"] = load("D_calibration")
save_report()

In [ ]:
# Phase E: molecular dynamics speed, Li3PS4 (2x2x1 supercell = 128 atoms), 800 K, normal precision.
# A short warm-up run absorbs the model compilation, then 300 steps are timed.
py("E_md", r"""
import json
from pathlib import Path
from pymatgen.core import Structure
from autolab.sim.backend import BudgetExceeded, relax_structures, run_md

s = Structure.from_dict(json.loads(Path("tests/data/Li3PS4_mp-985583.json").read_text()))
try:
    start, source = relax_structures([s], max_new=0)[0]["atoms"], "relaxée par UMA (cache)"
except BudgetExceeded:
    start, source = s, "structure MP non relaxée"
out = {"start": source, "temperature_K": 800}
for label, n_steps, seed in (("warmup", 20, 0), ("measure", 300, 1)):
    r = run_md(start, temperature_K=800, n_steps=n_steps, seed=seed)
    out[label] = {k: r[k] for k in ("n_atoms", "reps", "n_steps", "seconds", "ms_per_step", "cached")}
    print(label, out[label])
out["minutes_for_50ps"] = round(50_000 * out["measure"]["ms_per_step"] / 1000 / 60, 1)
open("/content/autolab_run/E_md.json", "w").write(json.dumps(out))
print("50 ps prendraient ≈", out["minutes_for_50ps"], "min sur ce GPU")
""")
REPORT["results"]["md"] = load("E_md")
save_report()

In [ ]:
# Final report (copy everything below) + zip of the results to download.
def junit(name):
    """{test name: (status, seconds)} from a pytest junit XML file."""
    import xml.etree.ElementTree as ET
    out = {}
    try:
        for tc in ET.parse(WORK / f"{name}.xml").iter("testcase"):
            status = "PASS"
            for tag, label in (("failure", "ÉCHEC"), ("error", "ERREUR"), ("skipped", "IGNORÉ")):
                if tc.find(tag) is not None:
                    status = label
            out[tc.get("name")] = (status, float(tc.get("time", 0)))
    except Exception:
        pass
    return out


def grab(log, pattern):
    try:
        m = re.search(pattern, (WORK / f"{log}.log").read_text(encoding="utf-8"))
        return m.group(1) if m else None
    except Exception:
        return None


lines = []
try:
    tests = {**junit("C_licl"), **junit("D_hull")}
    relax = relax_cache()
    licl = next((r for r in relax if set(r["atoms"]["numbers"]) == {3, 17}), None)
    a = None
    if licl:
        cell = licl["atoms"]["cell"]
        a = sum(sum(x * x for x in row) ** 0.5 for row in cell) / 3
    REPORT["results"]["licl_a"] = a

    def row(label, test, got, expected):
        status, secs = tests.get(test, ("NON LANCÉ", 0))
        lines.append(f"| {label} | {got if got is not None else '—'} | {expected} | {status} | {secs / 60:.1f} min |")

    lines += [
        "# Rapport étape 6 — UMA sur Colab",
        f"GPU : {REPORT.get('gpu')} | Python {REPORT.get('python')} | commit {REPORT.get('commit')}",
        f"Versions : {REPORT.get('versions')}",
        f"Secrets : {REPORT.get('secrets')}",
        "",
        "## Tests de vérité UMA (SPEC section 7)",
        "| Test | Obtenu | Attendu | Verdict | Durée |",
        "|---|---|---|---|---|",
    ]
    row("LiCl, paramètre de maille (Å)", "test_licl_relaxation",
        f"{a:.3f} (convergé {licl['converged']}, {licl['steps']} pas)" if licl else None,
        "5,10–5,18 (validé 5,145 ; exp. ≈ 5,13)")
    row("E_hull Li₂S (eV/atome)", "test_li2s_ehull_uma", grab("D_hull", r"Li2S E_hull \(UMA\) = (\S+?);"), "≤ 0,005")
    row("E_hull LiS rocksalt fictif", "test_fictitious_rocksalt_lis_is_unstable",
        grab("D_hull", r"Rocksalt LiS E_hull \(UMA\) = (\S+?),"), "≥ 0,3")
    row("E_hull Li₃PS₄ mp-985583", "test_li3ps4_calibration_uma",
        grab("D_hull", r"Li3PS4 mp-985583 E_hull \(UMA\) = (\S+)"), "≤ 0,01 (validé 0,000)")

    cal = REPORT["results"].get("calibration")
    lines += ["", "## Calibration Li-P-S (UMA vs MP, eV/atome)"]
    if cal:
        lines.append(f"{cal['n_refs']} références, écart moyen |UMA − MP| = {cal['mae_vs_mp']}, "
                     f"non convergées : {cal['not_converged'] or 'aucune'}")
        lines.append("SPEC : Li3PS4 0,000 / 0,007 / 0,015 (MP 0,000 / 0,007 / 0,014) ; Li7P3S11 0,010 (MP 0,020) ; Li2S 0")
        for d in cal["rows"]:
            if d["formula"] in ("Li3PS4", "Li7P3S11", "Li2S", "Li4P2S6", "Li7PS6", "Li3P", "P2S5"):
                lines.append(f"- {d['formula']} {d['id']} ({d['nsites']} at.) : UMA {d['uma']:.3f} | MP {d['mp']:.3f}")
    else:
        lines.append("non disponible (voir D_calibration.log)")

    lines += ["", "## Durées"]
    for step, info in REPORT["steps"].items():
        extra = {k: v for k, v in info.items() if k not in ("status", "seconds")}
        lines.append(f"- {step} : {info['status']}, {info['seconds'] / 60:.1f} min {extra or ''}")
    dl = REPORT["results"].get("download")
    if dl:
        lines.append(f"- checkpoint : {dl['size_gb']} Go en {dl['seconds']} s")
    if relax:
        relax.sort(key=lambda r: r["mtime"])
        secs = [r["seconds"] for r in relax]
        rest = sorted(secs[1:]) or [0]
        lines.append(f"- relaxations : {len(relax)} au total, {sum(secs) / 60:.1f} min ; la 1re (avec compilation) "
                     f"{secs[0]:.0f} s ; ensuite médiane {rest[len(rest) // 2]:.1f} s, max {max(rest):.0f} s ; "
                     f"non convergées : {sum(not r['converged'] for r in relax)}")
    mdr = REPORT["results"].get("md")
    if mdr:
        m = mdr["measure"]
        lines.append(f"- dynamique : {m['n_atoms']} atomes, {m['ms_per_step']} ms/pas (SPEC : 65 ms/pas pour 128 atomes) ; "
                     f"échauffement {mdr['warmup']['seconds']} s pour 20 pas ; 50 ps ≈ {mdr['minutes_for_50ps']} min ; "
                     f"départ : {mdr['start']}")
    lines.append(f"- durée totale du notebook : {(time.time() - T_START) / 60:.1f} min")
except Exception as e:
    lines.append(f"ERREUR pendant le rapport : {type(e).__name__}: {e}")

text = "\n".join(lines)
(WORK / "report.md").write_text(text, encoding="utf-8")
save_report()
print(text)

# Zip: UMA results cache, MP cache, logs and report (no checkpoint, no key).
try:
    stage = Path("/content/autolab_step6_results")
    shutil.rmtree(stage, ignore_errors=True)
    for sub in ("relax", "md", "mp"):
        if (CACHE / sub).exists():
            shutil.copytree(CACHE / sub, stage / "cache" / sub)
    shutil.copytree(WORK, stage / "run")
    archive = shutil.make_archive("/content/autolab_step6_results", "zip", stage)
    print(f"\nZip prêt : {archive} ({os.path.getsize(archive) / 1e6:.1f} Mo)")
    from google.colab import files
    files.download(archive)
except Exception as e:
    print(f"Zip/téléchargement : {type(e).__name__}: {e} — le fichier reste dans le panneau Fichiers de Colab.")